# LLM Inference Optimization & GPU Economics

**Track:** Model Deployment  
**Toolchain:** vLLM · GPTQ/AWQ · Flash Attention · KV-Cache · Speculative Decoding  
**Objective:** Learn how to serve LLMs 5-10x faster and 3-5x cheaper using modern inference optimization techniques.

---

## 🧠 Why Inference Optimization Matters More Than Training

You train a model ONCE. You serve it MILLIONS of times.

| Activity | Frequency | Cost Impact |
|----------|-----------|-------------|
| Training/Fine-tuning | Once per version (weekly/monthly) | One-time cost |
| Inference | Every user request (millions/day) | Ongoing, dominates budget |

**Real example:** A chatbot serving 1M requests/day at $0.01/request = **$300K/year** in inference costs. Optimizing to $0.003/request saves $200K/year.

### The Inference Cost Equation

```
Cost per request = (GPU cost per hour) / (Requests per hour per GPU)

To reduce cost, you either:
  1. Use cheaper GPUs (quantization lets models fit on smaller GPUs)
  2. Serve more requests per GPU (batching, caching, optimization)
  3. Reduce tokens generated (output length, early stopping)
```

---

## 📑 Table of Contents

* [🧠 Why Inference Optimization Matters More Than Training](#why-inference-optimization-matters-more-than-training)
  * [The Inference Cost Equation](#the-inference-cost-equation)
  * [🎓 Junior to Senior: Concept Bridge](#junior-to-senior-concept-bridge)
* [1 · Quantization: Shrink Models Without Losing Quality](#1-quantization-shrink-models-without-losing-quality)
  * [🤔 What is Quantization?](#what-is-quantization)
  * [Quantization Formats](#quantization-formats)
  * [🤔 When to Use Each?](#when-to-use-each)
* [2 · KV-Cache Optimization & Flash Attention](#2-kv-cache-optimization-flash-attention)
  * [🤔 What is the KV-Cache?](#what-is-the-kv-cache)
  * [KV-Cache Optimization Strategies](#kv-cache-optimization-strategies)
  * [🤔 What is Flash Attention?](#what-is-flash-attention)
* [3 · Batching & Continuous Batching](#3-batching-continuous-batching)
  * [🤔 Why Batching Matters](#why-batching-matters)
  * [Continuous Batching (vLLM's Secret)](#continuous-batching-vllms-secret)
* [4 · Speculative Decoding: Free Speed](#4-speculative-decoding-free-speed)
  * [🤔 How Does Speculative Decoding Work?](#how-does-speculative-decoding-work)
  * [⚖️ Trade-offs](#trade-offs)
* [Knowledge Check](#knowledge-check)
  * [Q1: Quantization Trade-off](#q1-quantization-trade-off)
  * [Q2: Continuous vs Static Batching](#q2-continuous-vs-static-batching)
  * [Q3: Speculative Decoding](#q3-speculative-decoding)
* [Practical Practice](#practical-practice)
  * [Exercise 1: A 13B model needs to fit on a 24GB GPU. Which quantization format would you choose and why?](#exercise-1-a-13b-model-needs-to-fit-on-a-24gb-gpu-which-quantization-format-would-you-choose-and-why)
  * [Exercise 2: Calculate the KV-cache memory for Llama 3 8B (32 layers, 4096 hidden, 32 heads) with batch=64 and seq_len=4096.](#exercise-2-calculate-the-kv-cache-memory-for-llama-3-8b-32-layers-4096-hidden-32-heads-with-batch64-and-seq_len4096)
* [5 · Edge & On-Device Inference](#5-edge-on-device-inference)
  * [🤔 When to Run Models Locally?](#when-to-run-models-locally)
  * [Edge Inference Tools](#edge-inference-tools)
* [🎯 Summary & Key Takeaways](#summary-key-takeaways)
  * [Priority Order](#priority-order)


### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** The true cost of LLMs isn't training; it's serving a million daily users. Seniors fiercely optimize inference using vLLM, continuous batching, and KV-cache management to squeeze 5x more throughput out of expensive GPUs.

**Mental Model:** Continuous batching is like an elevator. Instead of waiting for all people to reach the same floor, it lets new people in and out at every token generation step, maximizing occupancy.

**Common Junior Pitfall:** Deploying 16-bit generic huggingface `pipeline()` in production. PagedAttention (vLLM) and 4-bit/8-bit Quantization (AWQ/GPTQ) are strictly required to avoid out-of-memory errors on concurrent traffic.

---


In [ ]:
# Cell 1 — Install
!pip install -q numpy matplotlib

## 1 · Quantization: Shrink Models Without Losing Quality

### 🤔 What is Quantization?

Every model parameter is normally a 16-bit (fp16) or 32-bit (fp32) number. **Quantization** converts these to 8-bit or 4-bit numbers, shrinking the model 2-4x.

**Analogy:** Imagine a photo at 4K resolution (fp16). You can compress it to 720p (int8) and it still looks great. Compress to 240p (int4) and you notice artifacts, but it's still usable — and it loads 4x faster.

### Quantization Formats

| Format | Bits | Model Size Reduction | Quality Loss | Best For |
|--------|------|---------------------|-------------|----------|
| **fp16** | 16 | Baseline | None | Maximum quality |
| **int8** | 8 | 2x smaller | Minimal (<1%) | Good balance |
| **GPTQ** | 4 | 4x smaller | Small (1-3%) | GPU inference |
| **AWQ** | 4 | 4x smaller | Very small (0.5-2%) | GPU inference (better than GPTQ) |
| **GGUF** | 2-8 | 2-8x smaller | Varies | CPU inference (llama.cpp) |

### 🤔 When to Use Each?

| Scenario | Recommended | Why |
|----------|-------------|-----|
| Production server GPU | **AWQ 4-bit** | Best quality-to-compression ratio |
| Consumer GPU (16GB) | **GPTQ 4-bit** | Fits large models on small GPUs |
| CPU-only inference | **GGUF Q4** | Designed for CPU, via llama.cpp |
| Maximum quality needed | **fp16** | Medical, legal, safety-critical |
| Edge/mobile device | **GGUF Q2-Q4** | Smallest possible size |

In [ ]:
# Cell 2 — Quantization impact calculator
import numpy as np

def quantization_impact(model_params_b, original_bits=16):
    """Calculate memory savings and quality impact of quantization."""
    original_size_gb = model_params_b * original_bits / 8  # bytes, then GB
    
    quant_levels = [
        ('fp16 (baseline)', 16, 0.0),
        ('int8',            8,  0.5),
        ('GPTQ 4-bit',     4,  2.0),
        ('AWQ 4-bit',      4,  1.0),
        ('GGUF Q4',        4,  2.5),
        ('GGUF Q2',        2,  5.0),
    ]
    
    results = []
    for name, bits, quality_loss in quant_levels:
        size_gb = model_params_b * bits / 8
        reduction = 1 - (size_gb / original_size_gb)
        fits_16gb = '✅' if size_gb <= 16 else '❌'
        fits_24gb = '✅' if size_gb <= 24 else '❌'
        fits_80gb = '✅' if size_gb <= 80 else '❌'
        results.append({
            'format': name, 'size': f'{size_gb:.1f} GB', 
            'reduction': f'{reduction:.0%}', 'quality_loss': f'~{quality_loss}%',
            '16GB GPU': fits_16gb, '24GB GPU': fits_24gb, '80GB GPU': fits_80gb,
        })
    return results

for model_name, params in [('Llama 3 8B', 8), ('Llama 3 70B', 70), ('Mixtral 8x7B', 47)]:
    print(f'\n🔢 {model_name} ({params}B parameters)')
    print(f'{"Format":<18} {"Size":<10} {"Reduction":<12} {"Quality Loss":<14} {"16GB":<6} {"24GB":<6} {"80GB":<6}')
    print('-' * 72)
    for r in quantization_impact(params):
        print(f'{r["format"]:<18} {r["size"]:<10} {r["reduction"]:<12} {r["quality_loss"]:<14} {r["16GB GPU"]:<6} {r["24GB GPU"]:<6} {r["80GB GPU"]:<6}')

print(f'\n💡 Key insight: AWQ 4-bit gives the best quality-to-size ratio.')
print(f'   A 70B model goes from 140GB (needs 2x H100) to 35GB (fits on 1x A100)!')

---

## 2 · KV-Cache Optimization & Flash Attention

### 🤔 What is the KV-Cache?

When an LLM generates text, it computes **Key** and **Value** matrices for every token. Without caching, generating a 1000-token response would recompute all previous tokens 1000 times.

**The KV-Cache** stores these matrices so each new token only needs ONE computation step.

**Problem:** The KV-cache grows with sequence length and batch size. For a 70B model with 100 concurrent users at 4K context, the KV-cache alone uses **40-60 GB** of GPU memory.

### KV-Cache Optimization Strategies

| Strategy | How It Works | Memory Savings |
|----------|-------------|---------------|
| **PagedAttention (vLLM)** | Manages KV-cache like OS virtual memory pages | 2-4x more concurrent requests |
| **Prefix Caching** | Cache KV for shared system prompts | 30-60% savings on first tokens |
| **Sliding Window** | Only cache last N tokens | Fixed memory regardless of length |
| **Multi-Query Attention (MQA)** | Share K&V across attention heads | 8x less KV memory |

### 🤔 What is Flash Attention?

Normal attention reads/writes to GPU memory many times. **Flash Attention** reorganizes the computation to minimize memory reads, making it 2-4x faster.

**You don't implement Flash Attention** — you just enable it:
```python
model = AutoModelForCausalLM.from_pretrained(name, attn_implementation="flash_attention_2")
```

In [ ]:
# Cell 3 — KV-Cache memory calculator

def kv_cache_memory(model_params_b, num_layers, hidden_dim, num_heads,
                     seq_length, batch_size, dtype_bytes=2):
    """
    Calculate KV-cache memory usage.
    
    The KV cache stores Key and Value vectors for each layer,
    for each token in the sequence, for each request in the batch.
    """
    head_dim = hidden_dim // num_heads
    # Each layer stores K and V, each of shape: [batch, heads, seq_len, head_dim]
    kv_per_layer = 2 * batch_size * num_heads * seq_length * head_dim * dtype_bytes
    total_bytes = kv_per_layer * num_layers
    total_gb = total_bytes / (1024**3)
    return total_gb

# Common model architectures
models = [
    ('Llama 3 8B',   8,   32, 4096, 32, ),
    ('Llama 3 70B',  70,  80, 8192, 64, ),
    ('Mistral 7B',   7,   32, 4096, 32, ),
]

print('📊 KV-Cache Memory Usage')
print(f'{"Model":<16} {"Batch":<8} {"Seq Len":<10} {"KV-Cache":<12} {"With PagedAttention":<20}')
print('=' * 70)

for name, params, layers, hidden, heads in models:
    for batch_size, seq_len in [(1, 2048), (32, 2048), (32, 8192)]:
        kv_mem = kv_cache_memory(params, layers, hidden, heads, seq_len, batch_size)
        paged_mem = kv_mem * 0.4  # PagedAttention typically saves 60%
        print(f'{name:<16} {batch_size:<8} {seq_len:<10} {kv_mem:<12.1f} GB {paged_mem:<20.1f} GB')
    print()

print('💡 PagedAttention (vLLM) reduces KV-cache memory by ~60%, allowing')
print('   2-3x more concurrent requests on the same GPU.')

---

## 3 · Batching & Continuous Batching

### 🤔 Why Batching Matters

Without batching: 1 request at a time → GPU sits 90% idle.  
With batching: 32 requests at once → GPU fully utilized.

| Batching Strategy | How It Works | Throughput | Latency |
|-------------------|-------------|------------|--------|
| **No batching** | Process one request at a time | 🐌 Lowest | ⚡ Lowest |
| **Static batching** | Wait for N requests, process together | 🚗 Medium | 🔄 Higher (waiting) |
| **Continuous batching** | Add new requests as old ones finish | 🚀 Highest | ⚡ Low (no waiting) |

### Continuous Batching (vLLM's Secret)

Static batching waits until a batch is full. Continuous batching adds new requests to the GPU AS SOON as a slot opens up — no waiting.

```
Static Batching:    [Req1, Req2, Req3, Req4] → Process → [Req5, Req6, Req7, Req8]
                    └── Wait until all done ──┘          └── Wait again ──────────┘

Continuous Batching: [Req1, Req2, Req3, Req4] → Req1 done → [Req5, Req2, Req3, Req4]
                                                  ↑ Slot freed, Req5 immediately starts!
```

In [ ]:
# Cell 4 — Batching throughput simulator
import numpy as np

def simulate_batching(num_requests, strategy='none', max_batch_size=32):
    """Simulate different batching strategies."""
    np.random.seed(42)
    # Each request takes 0.5-3 seconds to generate
    request_times = np.random.uniform(0.5, 3.0, num_requests)
    
    if strategy == 'none':
        total_time = sum(request_times)  # Sequential
    elif strategy == 'static':
        batches = [request_times[i:i+max_batch_size] for i in range(0, num_requests, max_batch_size)]
        total_time = sum(max(batch) for batch in batches)  # Slowest in each batch
    else:  # continuous
        # Continuous batching starts new requests as old ones finish
        # Approximate: total_time ≈ sum(times) / batch_size + overhead
        total_time = sum(request_times) / max_batch_size * 1.1  # 10% overhead
    
    throughput = num_requests / total_time
    avg_latency = total_time / num_requests
    return {'time': total_time, 'throughput': throughput, 'avg_latency': avg_latency}

num_reqs = 100
print(f'⚡ Batching Strategy Comparison ({num_reqs} requests)')
print(f'{"Strategy":<22} {"Total Time":<14} {"Throughput":<18} {"Avg Latency":<14}')
print('=' * 68)

for strategy in ['none', 'static', 'continuous']:
    r = simulate_batching(num_reqs, strategy)
    name = {'none': 'No batching', 'static': 'Static (batch=32)', 'continuous': 'Continuous (vLLM)'}[strategy]
    print(f'{name:<22} {r["time"]:>8.1f} sec   {r["throughput"]:>8.1f} req/sec   {r["avg_latency"]:>8.2f} sec')

print(f'\n💡 Continuous batching (used by vLLM) achieves 10-30x higher throughput')
print(f'   than processing one request at a time.')

---

## 4 · Speculative Decoding: Free Speed

### 🤔 How Does Speculative Decoding Work?

**Problem:** Large models are slow because they generate ONE token at a time.  
**Idea:** Use a SMALL, FAST model to "draft" 5-10 tokens, then have the LARGE model verify them in ONE pass.

```
Normal:       [Large Model] → token1 → token2 → token3 → token4 → token5
              5 sequential steps × 100ms each = 500ms

Speculative:  [Small Model] → draft: token1, token2, token3, token4, token5
              [Large Model] → verify all 5 at once (1 forward pass)
              → Accept tokens 1-3 (correct), reject 4-5 (wrong)
              → Regenerate from token 4
              ~2 steps × 100ms = 200ms (2.5x faster!)
```

### ⚖️ Trade-offs

| Aspect | Benefit | Cost |
|--------|---------|------|
| Speed | 2-3x faster generation | Extra GPU memory for draft model |
| Quality | IDENTICAL to large model (mathematically proven) | Slightly more complex setup |
| Best for | Long outputs (> 100 tokens) | Not helpful for short outputs |

In [ ]:
# Cell 5 — Speculative decoding simulation
import numpy as np

def simulate_speculative_decoding(total_tokens, draft_model_acceptance_rate=0.7, 
                                   draft_batch_size=5, large_model_ms=100, draft_model_ms=10):
    """Simulate speculative decoding speedup."""
    
    # Normal decoding: one token at a time
    normal_time = total_tokens * large_model_ms
    
    # Speculative decoding
    spec_time = 0
    tokens_generated = 0
    np.random.seed(42)
    
    while tokens_generated < total_tokens:
        # Draft model generates a batch of tokens (fast)
        spec_time += draft_model_ms * draft_batch_size
        
        # Large model verifies entire batch (one forward pass)
        spec_time += large_model_ms
        
        # How many tokens are accepted?
        accepted = 0
        for _ in range(draft_batch_size):
            if np.random.random() < draft_model_acceptance_rate:
                accepted += 1
            else:
                break  # First rejection stops the chain
        
        tokens_generated += max(1, accepted)  # At least 1 token per step
    
    speedup = normal_time / spec_time
    return {
        'normal_ms': normal_time,
        'speculative_ms': spec_time,
        'speedup': speedup,
    }

print('🚀 Speculative Decoding Speedup')
print(f'{"Output Length":<16} {"Normal":<14} {"Speculative":<14} {"Speedup":<10}')
print('=' * 55)

for tokens in [50, 200, 500, 1000]:
    r = simulate_speculative_decoding(tokens)
    print(f'{tokens} tokens      {r["normal_ms"]/1000:>8.1f}s     {r["speculative_ms"]/1000:>8.1f}s     {r["speedup"]:>5.1f}x')

print(f'\n💡 Speculative decoding is most effective for long outputs.')
print(f'   The quality is IDENTICAL to the large model (no approximation).')

In [ ]:
# Cell 6 -- GPU economics calculator
import numpy as np

def gpu_cost_analysis(requests_per_day, tokens_per_request=200,
                      gpu_cost_per_hour=2.0):
    configs = [
        ('No optimization (HF pipeline)', 30, 16),
        ('vLLM + fp16', 200, 16),
        ('vLLM + AWQ 4-bit', 400, 8),
        ('vLLM + AWQ + speculative', 600, 8),
    ]
    print(f'Daily requests: {requests_per_day:,}')
    print(f'{"Config":<35} {"TPS":<8} {"GPUs":<6} {"$/day":<10} {"$/month"}')
    print('=' * 75)
    for name, tps, gpu_gb in configs:
        tokens_per_day = requests_per_day * tokens_per_request
        tokens_per_sec_needed = tokens_per_day / 86400
        gpus = max(1, int(np.ceil(tokens_per_sec_needed / tps)))
        daily_cost = gpus * gpu_cost_per_hour * 24
        monthly = daily_cost * 30
        print(f'{name:<35} {tps:<8} {gpus:<6} ${daily_cost:<9.0f} ${monthly:,.0f}')

gpu_cost_analysis(100_000)
print('\nKey: vLLM + AWQ 4-bit is the sweet spot for most production use cases.')


---
## Knowledge Check

### Q1: Quantization Trade-off
A medical AI team says 'We must use fp16, no quantization.' Are they right? When is this justified?

<details><summary>Answer</summary>

Yes, they may be right. Medical/legal/safety-critical applications where even 1% quality loss could harm patients may justify the 4x higher cost of fp16. For most other use cases (chatbots, summarization, code), AWQ 4-bit with <1% quality loss is a better trade-off.
</details>

### Q2: Continuous vs Static Batching
You have 4 requests: A(100 tokens), B(500 tokens), C(50 tokens), D(200 tokens). With static batching, when does C finish?

<details><summary>Answer</summary>

With static batching, ALL requests in the batch must wait for the SLOWEST one (B: 500 tokens). So C finishes after ~500 token-generation cycles, even though it only needed 50. With continuous batching, C finishes early and its slot is immediately given to a new request.
</details>

### Q3: Speculative Decoding
Your draft model has 90% acceptance rate. Is this good or bad?

<details><summary>Answer</summary>

90% is excellent! Each draft batch of 5 tokens will accept ~4.5 on average, meaning the large model only runs once to verify 5 tokens instead of 5 separate forward passes. Expected speedup: ~3-4x. If acceptance drops below 50%, speculative decoding becomes slower than normal.
</details>


---
## Practical Practice

### Exercise 1: A 13B model needs to fit on a 24GB GPU. Which quantization format would you choose and why?
### Exercise 2: Calculate the KV-cache memory for Llama 3 8B (32 layers, 4096 hidden, 32 heads) with batch=64 and seq_len=4096.


In [ ]:
# ===== SOLUTIONS =====
print('Ex 1: 13B model on 24GB GPU')
for fmt, bits, loss in [('fp16', 16, '0%'), ('int8', 8, '<1%'), ('AWQ 4-bit', 4, '~1%')]:
    size = 13 * bits / 8
    fits = 'YES' if size < 22 else 'NO'  # Leave 2GB headroom
    print(f'  {fmt}: {size:.0f}GB - Fits: {fits} (quality loss: {loss})')
print('  Best: AWQ 4-bit (6.5GB, leaves 17.5GB for KV-cache + runtime)')

print('\nEx 2: KV-cache for Llama 3 8B')
layers, hidden, heads = 32, 4096, 32
batch, seq = 64, 4096
head_dim = hidden // heads
kv_bytes = 2 * batch * heads * seq * head_dim * 2 * layers  # 2=K+V, 2=fp16
kv_gb = kv_bytes / (1024**3)
print(f'  head_dim = {hidden}/{heads} = {head_dim}')
print(f'  KV-cache = 2 * {batch} * {heads} * {seq} * {head_dim} * 2 bytes * {layers} layers')
print(f'  = {kv_gb:.1f} GB')
print(f'  This is MORE than the model weights (16GB)!')


---

## 5 · Edge & On-Device Inference

### 🤔 When to Run Models Locally?

| Scenario | Edge/Local | Cloud |
|----------|-----------|-------|
| Offline required (airplane, rural) | ✅ | ❌ |
| Ultra-low latency (< 10ms) | ✅ | ❌ |
| Privacy-critical (medical records on device) | ✅ | ❌ |
| Large model (70B+) | ❌ | ✅ |
| Always up-to-date model | ❌ | ✅ |

### Edge Inference Tools

| Tool | Platform | Models | Best For |
|------|---------|--------|----------|
| **llama.cpp** | CPU/GPU (any OS) | GGUF format | Desktop/server, broadest support |
| **Ollama** | Desktop wrapper | Any GGUF | Developer experience |
| **ONNX Runtime** | Cross-platform | ONNX format | Production edge deployment |
| **TensorFlow Lite** | Mobile/IoT | TFLite format | Android, embedded |
| **Core ML** | Apple devices | Core ML format | iOS, macOS |

**2026 reality:** A Phi-3 Mini (3.8B) model runs at 20+ tokens/sec on a modern laptop CPU. Small models on edge are practical today.

---

## 🎯 Summary & Key Takeaways

| Technique | Speedup | Cost Savings | Complexity |
|-----------|---------|-------------|------------|
| AWQ/GPTQ Quantization | 1.5-2x | 2-4x (smaller GPU) | Low |
| Flash Attention | 2-4x | Same GPU, faster | Low (one flag) |
| Continuous Batching (vLLM) | 10-30x throughput | 5-10x per-request | Medium |
| PagedAttention | 2-3x more concurrent users | 2-3x | Low (built into vLLM) |
| Speculative Decoding | 2-3x latency | Same cost, faster | Medium |
| Prefix Caching | 30-60% first-token | 30-60% for system prompts | Low |

### Priority Order

1. **Use vLLM** (gets you continuous batching + PagedAttention for free)
2. **Enable Flash Attention** (one flag, 2-4x faster)
3. **Quantize to AWQ 4-bit** (if model doesn't fit on your GPU)
4. **Enable speculative decoding** (if generating long outputs)
5. **Prefix caching** (if all requests share a system prompt)

**Next →** `31_on_device_edge_llms.ipynb` — On-Device & Edge LLMs